# C3.2 Practice — Make Your HDB Model Better & More Trustworthy
## By TFY

**Module 3 · Machine Learning & GenAI — Coaching Add-on**

In [1]:
# Setup — run this first (needs internet to download the data)
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

url = ("https://raw.githubusercontent.com/flexfengfeng/6m-data-C3.2/"
       "main/notebooks/data/"
       "Resale_flat_prices_based_on_registration_date_from_Jan-2017_onwards.csv")
data = pd.read_csv(url)
data["floor_level"] = data["storey_range"].str.split(" ").str[0].astype(float)
# 
print("Rows of real HDB sales loaded:", len(data))
data.head()

Rows of real HDB sales loaded: 233479


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,floor_level
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,10.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,1.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,1.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,4.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,1.0


> 📊 **Reading the output:** 

In [2]:
print("Data types:")
print(data.dtypes)
print()
print("Missing values:")
missing = data.isna().sum()
print(missing[missing > 0])
print()
print("Numerical summary (excluding missing):")
print(data.describe(include="number").round(2))

Data types:
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
remaining_lease         object
resale_price           float64
floor_level            float64
dtype: object

Missing values:
Series([], dtype: int64)

Numerical summary (excluding missing):
       floor_area_sqm  lease_commence_date  resale_price  floor_level
count       233479.00            233479.00     233479.00    233479.00
mean            96.69              1996.55     531028.39         7.78
std             24.02                14.36     190174.93         5.95
min             31.00              1966.00     140000.00         1.00
25%             81.00              1985.00     390000.00         4.00
50%             93.00              1997.00     500000.00         7.00
75%            112.00         

## 📏 Three scorecards, in plain English (no stats needed)

From here on we judge every model with three numbers. Here's what they really mean:

**MAE — "on average, how many dollars off?"**

**MAPE — "on average, how many *percent* off?"**

**R² — "out of everything that makes prices differ, how much did the model figure out?"**

### ✋ Pause & Predict


In [3]:
# Part A — baseline model with the original 3 features
features_3 = ["floor_area_sqm", "lease_commence_date", "floor_level"]
X = data[features_3]
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = LinearRegression()
baseline.fit(X_train, y_train)
preds = baseline.predict(X_test)

mae_baseline = mean_absolute_error(y_test, preds)
mape_baseline = mean_absolute_percentage_error(y_test, preds)
r2_baseline = r2_score(y_test, preds)
print(f"Baseline (3 features): on average off by S${mae_baseline:,.0f}")
print(f"Baseline (3 features): MAPE = {mape_baseline:.1%}  (average % off — fair across cheap & pricey flats)")
print(f"Baseline (3 features): R² = {r2_baseline:.3f}  (share of price variation explained, 1.0 = perfect)")

Baseline (3 features): on average off by S$102,499
Baseline (3 features): MAPE = 20.1%  (average % off — fair across cheap & pricey flats)
Baseline (3 features): R² = 0.500  (share of price variation explained, 1.0 = perfect)


## Part B — 🔧 YOUR TURN: Add more features


In [4]:
# Part B — add flat_type and town
numeric = ["floor_area_sqm", "lease_commence_date", "floor_level"]
category = ["flat_type", "town"]

X_more = pd.get_dummies(data[numeric + category], columns=category)   
 
y = data["resale_price"]


Step 1 — It identifies all unique values in each categorical column
For flat_type, that's something like: 1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, etc.

For town, that's something like: ANG MO KIO, BEDOK, BISHAN, etc. (likely 20+ towns).

Step 2 — It creates one new column per unique value
Each new column is named <original_column>_<value>, e.g.:

flat_type_2 ROOM, flat_type_3 ROOM, flat_type_4 ROOM, ...
town_ANG MO KIO, town_BEDOK, town_BISHAN, ...

Step 3 — It fills each new column with 1 where that row matches, 0 otherwise
So a row where flat_type == "3 ROOM" and town == "ANG MO KIO" gets 1 in flat_type_3 ROOM and town_ANG MO KIO, and 0 in every other dummy column for those two features.

Step 4 — The original flat_type and town object columns are dropped entirely
They're replaced by the new 0/1 columns — that's why the object dtype disappears from your dataframe once X_more is built.

#### One important nuance — check your actual dtype:

In recent pandas versions (≥1.5 or so, including what you're likely running), pd.get_dummies() by default returns these new columns as bool dtype (True/False), not int64. This matters because:

In [5]:
print(X_more.dtypes)

floor_area_sqm                float64
lease_commence_date             int64
floor_level                   float64
flat_type_1 ROOM                 bool
flat_type_2 ROOM                 bool
flat_type_3 ROOM                 bool
flat_type_4 ROOM                 bool
flat_type_5 ROOM                 bool
flat_type_EXECUTIVE              bool
flat_type_MULTI-GENERATION       bool
town_ANG MO KIO                  bool
town_BEDOK                       bool
town_BISHAN                      bool
town_BUKIT BATOK                 bool
town_BUKIT MERAH                 bool
town_BUKIT PANJANG               bool
town_BUKIT TIMAH                 bool
town_CENTRAL AREA                bool
town_CHOA CHU KANG               bool
town_CLEMENTI                    bool
town_GEYLANG                     bool
town_HOUGANG                     bool
town_JURONG EAST                 bool
town_JURONG WEST                 bool
town_KALLANG/WHAMPOA             bool
town_MARINE PARADE               bool
town_PASIR R

---

#### Note: can I instead use this code "data[category] = data[caterhgory].astype(bool)" to convert?

- No — that line does something completely different from what you need, and it would break your model. Let me explain why.
- What data[category] = data[category].astype(bool) actually does:
- It takes each string value in flat_type and town and converts it to True or False based on Python's truthiness rule for strings — and for non-empty strings, every single value becomes True. The only way you'd get False is if the cell were an empty string "" or NaN.

So if you ran this, you'd end up with:
       
        flat_type (before)             flat_type (after)
           "2 ROOM"                         True
           "3 ROOM"                         True
           "EXECUTIVE"                      True

---

In [6]:
# Continue from above, train/test split, fit, predict, and evaluate the richer model
X_train, X_test, y_train, y_test = train_test_split(X_more, y, test_size=0.2, random_state=42)

richer = LinearRegression()
richer.fit(X_train, y_train)
richer_preds = richer.predict(X_test)
mae_richer = mean_absolute_error(y_test, richer_preds)
mape_richer = mean_absolute_percentage_error(y_test, richer_preds)
r2_richer = r2_score(y_test, richer_preds)

print(f"Baseline (3 features): MAE S${mae_baseline:,.0f}   MAPE {mape_baseline:.1%}   R² {r2_baseline:.3f}")
print(f"Richer  (5 features): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Improvement: S${mae_baseline - mae_richer:,.0f} smaller error, MAPE {mape_baseline - mape_richer:+.1%}, R² up {r2_richer - r2_baseline:+.3f}")

Baseline (3 features): MAE S$102,499   MAPE 20.1%   R² 0.500
Richer  (5 features): MAE S$82,761   MAPE 16.7%   R² 0.704
Improvement: S$19,738 smaller error, MAPE +3.4%, R² up +0.205


## Part C — 🔧 YOUR TURN: Choose a model + check for overfitting


In [7]:
# Part C — compare Linear Regression vs Random Forest
# 🔧 YOUR TURN: create a Random Forest model in the blank.
forest = RandomForestRegressor(n_estimators=100, random_state=42)   # <- fill the blank
forest.fit(X_train, y_train)

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test   = mean_absolute_error(y_test,  forest.predict(X_test))
mae_train  = mean_absolute_error(y_train, forest.predict(X_train))
mape_test  = mean_absolute_percentage_error(y_test,  forest.predict(X_test))
mape_train = mean_absolute_percentage_error(y_train, forest.predict(X_train))
r2_test    = r2_score(y_test,  forest.predict(X_test))
r2_train   = r2_score(y_train, forest.predict(X_train))

print(f"Linear Regression  test: MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Random Forest      test: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print()
print(f"Random Forest  TRAIN: MAE S${mae_train:,.0f}   MAPE {mape_train:.1%}   R² {r2_train:.3f}")
print(f"Random Forest   TEST: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gap (overfitting signal): MAE S${mae_test - mae_train:,.0f}, MAPE {mape_test - mape_train:+.1%}, R² {r2_train - r2_test:+.3f}")

Linear Regression  test: MAE S$82,761   MAPE 16.7%   R² 0.704
Random Forest      test: MAE S$72,625   MAPE 14.4%   R² 0.774

Random Forest  TRAIN: MAE S$62,456   MAPE 12.6%   R² 0.833
Random Forest   TEST: MAE S$72,625   MAPE 14.4%   R² 0.774
Gap (overfitting signal): MAE S$10,169, MAPE +1.8%, R² +0.059


## 🏆 Challenge (if you have time)
Pick **one**:
1. Add a third feature of your choice (e.g. `remaining_lease` cleaned to a number) and see if the error drops further.
2. Try `RandomForestRegressor(n_estimators=300)` — does more trees help the test error, or just slow it down?
3. Find the single flat in the test set where the model was **most wrong**. What was special about it?

___
---
### (My input) Further improve MAE, MAPE, R2 using Hyperparameter.



In [8]:
# Part C — compare Linear Regression vs Random Forest
# 🔧 YOUR TURN: create a Random Forest model in the blank.

forest = RandomForestRegressor(
    n_estimators=300,
    max_features=0.6,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1          # parallelize across cores — same result, much faster
)

forest.fit(X_train, y_train)

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test   = mean_absolute_error(y_test,  forest.predict(X_test))
mae_train  = mean_absolute_error(y_train, forest.predict(X_train))
mape_test  = mean_absolute_percentage_error(y_test,  forest.predict(X_test))
mape_train = mean_absolute_percentage_error(y_train, forest.predict(X_train))
r2_test    = r2_score(y_test,  forest.predict(X_test))
r2_train   = r2_score(y_train, forest.predict(X_train))

print()
print(f"Random Forest  TRAIN: MAE S${mae_train:,.0f}   MAPE {mape_train:.1%}   R² {r2_train:.3f}")
print(f"Random Forest   TEST: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gap (overfitting signal): MAE S${mae_test - mae_train:,.0f}, MAPE {mape_test - mape_train:+.1%}, R² {r2_train - r2_test:+.3f}")


Random Forest  TRAIN: MAE S$66,333   MAPE 13.2%   R² 0.817
Random Forest   TEST: MAE S$70,963   MAPE 14.1%   R² 0.790
Gap (overfitting signal): MAE S$4,630, MAPE +0.8%, R² +0.027


### To push MAE/MAPE/R² further

#### Small hyperparameter tweaks (cheap, try these first)


In [9]:
forest = RandomForestRegressor(
    n_estimators=400,
    max_features=0.6,
    min_samples_leaf=3,      # slightly less aggressive than 5 — lets it capture more nuance
    max_depth=25,             # caps tree depth, extra guard against overfitting
    random_state=42,
    n_jobs=-1
)

forest.fit(X_train, y_train)

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test   = mean_absolute_error(y_test,  forest.predict(X_test))
mae_train  = mean_absolute_error(y_train, forest.predict(X_train))
mape_test  = mean_absolute_percentage_error(y_test,  forest.predict(X_test))
mape_train = mean_absolute_percentage_error(y_train, forest.predict(X_train))
r2_test    = r2_score(y_test,  forest.predict(X_test))
r2_train   = r2_score(y_train, forest.predict(X_train))

print()
print(f"Random Forest  TRAIN: MAE S${mae_train:,.0f}   MAPE {mape_train:.1%}   R² {r2_train:.3f}")
print(f"Random Forest   TEST: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gap (overfitting signal): MAE S${mae_test - mae_train:,.0f}, MAPE {mape_test - mape_train:+.1%}, R² {r2_train - r2_test:+.3f}")


Random Forest  TRAIN: MAE S$66,851   MAPE 13.4%   R² 0.813
Random Forest   TEST: MAE S$71,060   MAPE 14.1%   R² 0.788
Gap (overfitting signal): MAE S$4,209, MAPE +0.7%, R² +0.025


#### Let sklearn search for you (more rigorous, uses cross-validation instead of just eyeballing train/test):

In [10]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "max_features": [0.4, 0.5, 0.6, 0.8],
    "min_samples_leaf": [2, 3, 5, 8],
    "max_depth": [15, 20, 25, None],
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)
print(search.best_params_)
best_forest = search.best_estimator_

/home/mille/miniconda3/envs/dsai-m3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': None}


In [11]:

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test   = mean_absolute_error(y_test,  search.predict(X_test))
mae_train  = mean_absolute_error(y_train, search.predict(X_train))
mape_test  = mean_absolute_percentage_error(y_test,  search.predict(X_test))
mape_train = mean_absolute_percentage_error(y_train, search.predict(X_train))
r2_test    = r2_score(y_test,  search.predict(X_test))
r2_train   = r2_score(y_train, search.predict(X_train))

print()
print(f"CV Search  TRAIN: MAE S${mae_train:,.0f}   MAPE {mape_train:.1%}   R² {r2_train:.3f}")
print(f"CV Search   TEST: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gap (overfitting signal): MAE S${mae_test - mae_train:,.0f}, MAPE {mape_test - mape_train:+.1%}, R² {r2_train - r2_test:+.3f}")


CV Search  TRAIN: MAE S$66,459   MAPE 13.3%   R² 0.816
CV Search   TEST: MAE S$70,920   MAPE 14.1%   R² 0.790
Gap (overfitting signal): MAE S$4,461, MAPE +0.8%, R² +0.026


### Step 1: Clean remaining_lease into a number. It's a string like "61 years 04 months", so it needs parsing (the same regex approach from your HDB coursework):

In [ ]:
import re

def parse_lease(s):
    years = re.search(r"(\d+)\s*year", s)
    months = re.search(r"(\d+)\s*month", s)
    y = int(years.group(1)) if years else 0
    m = int(months.group(1)) if months else 0
    return y + m/12

data["remaining_lease_years"] = data["remaining_lease"].apply(parse_lease)

### Step 2: Add it to X_more. Since it's already numeric, it just goes in the numeric list:

In [ ]:
numeric = ["floor_area_sqm", "lease_commence_date", "remaining_lease_years", "floor_level"]
category = ["flat_type", "town"]

X_more = pd.get_dummies(data[numeric + category], columns=category)
y = data["resale_price"]

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_grid_classic = {
    "n_estimators":  [100, 200, 400],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth":     [3, 4, 6],
    "subsample":     [0.8, 1.0],
}
# 3 x 3 x 3 x 2 = 54 combos x 5-fold = 270 model fits — expect this to be noticeably slower

---
---

## Part D — 🔧 (Stretch) Stacking: can a *team* of models beat the best one?

Your learner asked a great question: can we combine Random Forest and Gradient Boosting?

In [8]:
# Part D — stack Random Forest + Gradient Boosting, blended by a Linear Regression "manager"
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

# Two different base models (they make different kinds of mistakes)
base_models = [
    ("random_forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("grad_boost",    GradientBoostingRegressor(random_state=42)),
]

# 🔧 YOUR TURN: the "manager" that learns how to blend the base models.
#    Use a simple LinearRegression() as the final_estimator.
stack = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),          # <- fill the blank
)
stack.fit(X_train, y_train)
stack_preds = stack.predict(X_test)
mae_stack  = mean_absolute_error(y_test, stack_preds)
mape_stack = mean_absolute_percentage_error(y_test, stack_preds)
r2_stack   = r2_score(y_test, stack_preds)

# Also score Gradient Boosting on its own, for a fair comparison
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
mae_gb  = mean_absolute_error(y_test, gb_preds)
mape_gb = mean_absolute_percentage_error(y_test, gb_preds)
r2_gb   = r2_score(y_test, gb_preds)

print(f"Linear Regression (Part B): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Random Forest    (Part C): MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gradient Boosting (alone) : MAE S${mae_gb:,.0f}   MAPE {mape_gb:.1%}   R² {r2_gb:.3f}")
print(f"Stacked team             : MAE S${mae_stack:,.0f}   MAPE {mape_stack:.1%}   R² {r2_stack:.3f}")

Linear Regression (Part B): MAE S$82,761   MAPE 16.7%   R² 0.704
Random Forest    (Part C): MAE S$72,625   MAPE 14.4%   R² 0.774
Gradient Boosting (alone) : MAE S$78,854   MAPE 15.7%   R² 0.725
Stacked team             : MAE S$71,900   MAPE 14.3%   R² 0.781


## Part E — 💡 So how *do* we push the error down further?

A learner asked the natural follow-up: *"We've tried more features, a Random Forest, and stacking — what else can we do?"*



In [9]:
# Part E — push further: (1) engineer a new feature, (2) tune the model

# --- Lever 1: feature engineering --------------------------------------------
# 'remaining_lease' is text like "61 years 04 months". Turn it into a single
# number of years (months count as a fraction) so the model can use it.
def lease_to_years(text):
    parts = text.split()
    years = int(parts[0])
    months = int(parts[2]) if "month" in text else 0
    return years + months / 12

data["remaining_lease_years"] = data["remaining_lease"].apply(lease_to_years)

# Same features as Part B, plus our new engineered one.
numeric_plus = ["floor_area_sqm", "lease_commence_date", "floor_level", "remaining_lease_years"]
category = ["flat_type", "town"]
X_plus = pd.get_dummies(data[numeric_plus + category], columns=category)
y = data["resale_price"]

# IMPORTANT: same random_state=42 split as before, so the comparison is fair.
X_train, X_test, y_train, y_test = train_test_split(X_plus, y, test_size=0.2, random_state=42)

# --- Lever 2: a more carefully tuned Gradient Boosting model -----------------
# More trees + a smaller learning_rate = many small, careful correction steps.
# HistGradientBoostingRegressor uses max_iter instead of n_estimators because n_estimators doesn't exist as a parameter
# GradientBoostingRegressor → use n_estimators instead of max_iter, gb is a classic model.
tuned_gb = GradientBoostingRegressor(
    n_estimators=400,      # more correction steps (default is 100)
    learning_rate=0.05,    # smaller steps generalise better
    max_depth=4,           # slightly deeper trees catch feature combinations
    random_state=42,
)
tuned_gb.fit(X_train, y_train)
tuned_preds = tuned_gb.predict(X_test)

mae_tuned  = mean_absolute_error(y_test, tuned_preds)
mape_tuned = mean_absolute_percentage_error(y_test, tuned_preds)
r2_tuned   = r2_score(y_test, tuned_preds)

print("Best so far (Random Forest, Part C):")
print(f"    MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print("Tuned Gradient Boosting + extra feature (Part E):")
print(f"    MAE S${mae_tuned:,.0f}   MAPE {mape_tuned:.1%}   R² {r2_tuned:.3f}")
print()
print(f"Change vs best so far: MAE S${mae_test - mae_tuned:,.0f} smaller, "
      f"MAPE {mape_tuned - mape_test:+.1%}, R² {r2_tuned - r2_test:+.3f}")

Best so far (Random Forest, Part C):
    MAE S$72,625   MAPE 14.4%   R² 0.774
Tuned Gradient Boosting + extra feature (Part E):
    MAE S$42,803   MAPE 8.3%   R² 0.908

Change vs best so far: MAE S$29,822 smaller, MAPE -6.0%, R² +0.134


## Part F — 💾 How big is each model when you save it as a `.pkl`?

A learner asked: *"If I save every model in this notebook to a `.pkl` file, how big is each one — and why are they so different?"*

In [10]:
# Part F — measure the saved (.pkl) size of every model we built
import pickle

def pkl_size(model):
    """Bytes this model takes when pickled (same as a .pkl file on disk)."""
    return len(pickle.dumps(model))

def human(n):
    for unit in ("B", "KB", "MB"):
        if n < 1024 or unit == "MB":
            return f"{n:,.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024

models = [
    ("A. Linear Regression (3 features)", baseline),
    ("B. Linear Regression (5 feat + dummies)", richer),
    ("C. Random Forest (100 trees)", forest),
    ("D. Gradient Boosting (100 steps, default)", gb),
    ("D. Stacking (RF + GB + LR manager)", stack),
    ("E. Tuned Gradient Boosting (400 steps, depth 4)", tuned_gb),
]

rows = []
for name, m in models:
    b = pkl_size(m)
    rows.append({"Model": name, "Bytes": b, "Size": human(b)})

sizes = pd.DataFrame(rows).sort_values("Bytes").reset_index(drop=True)
sizes

,Model,Bytes,Size
0,A. Linear Regression (3 features),607,607 B
1,B. Linear Regression (5 feat + dummies),1743,1.7 KB
2,"D. Gradient Boosting (100 steps, default)",139935,136.7 KB
3,"E. Tuned Gradient Boosting (400 steps, depth 4)",996873,973.5 KB
4,C. Random Forest (100 trees),343875998,327.9 MB
5,D. Stacking (RF + GB + LR manager),344015509,328.1 MB


## ✅ Wrap-up

| Idea | In one line |
|---|---|
| **MAE** | Average dollar error — your model's report card. |
| **Train/test split** | Test on unseen flats for an honest score. |
| **Adding features** | More relevant info usually shrinks the error. |
| **Model choice** | Pick the lowest **test** error, not training error. |
| **Overfitting** | Big train-vs-test gap = memorising, not learning. |

**Up next:** plug your better model into the upgraded `app.py` and redeploy in Streamlit.

---
## Sample solutions (look only after trying!)

**Part B blank:** `X_more = pd.get_dummies(data[numeric + category], columns=category)`

**Part C blank:** `forest = RandomForestRegressor(n_estimators=100, random_state=42)`


**Part D blank:** `final_estimator=LinearRegression()`

---


---